# 02 — Feature Engineering

**Objectif** : Préparer les données pour la modélisation. Enrichir le dataset 
avec des facteurs CO2 variables depuis la Base Carbone ADEME pour remplacer 
les constantes de Back-on-Track 2022.

**Inputs** : Base PostgreSQL `obrail_db`, `data/raw/Base_Carbone_V23.9.csv`  
**Outputs** : Dataset enrichi avec facteurs CO2 variables  
**Date** : Avril 2026

## Imports

In [6]:
import pandas as pd
from sqlalchemy import create_engine

## Chargement des données

In [7]:
df = pd.read_csv(
    '/Users/louisgardet/dev/python/OBRail-MSPR2/data/raw/environmental_impact.csv'
)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (3678, 18)


,route_name,origin,destination,service_type,route_name_simple,origin_country,destination_country,distance_km,operator,type,train_gco2_pkm,plane_gco2_pkm,train_co2_kg,plane_co2_kg,co2_savings_kg,savings_percent,emission_source,calculation_date
0,ICE 18,Hauptbahnhof,Berlin Gesundbrunnen,day,→ Berlin Gesundbrunnen,DE,DE,238.64,DB,day,14,144,3.34,34.36,31.02,90.3,Back-on-Track 2022,2026-03-22
1,IC 51,Hauptbahnhof,Dortmund Hbf,day,→ Dortmund,DE,DE,256.33,DB,day,14,144,3.59,36.91,33.32,90.3,Back-on-Track 2022,2026-03-22
2,ICE 28,Hamburg Altona S,Hauptbahnhof,day,Hamburg Altona S →,DE,DE,296.60,DB,day,14,144,4.15,42.71,38.56,90.3,Back-on-Track 2022,2026-03-22


## Extraction des facteurs CO2 ferroviaires (ADEME Base Carbone V23.9)

La Base Carbone contient 23 000+ entrées. On filtre uniquement les lignes 
transport ferroviaire avec une unité par passager-km.

In [8]:
df_ademe = pd.read_csv(
    '/Users/louisgardet/dev/python/OBRail-MSPR2/data/raw/Base_Carbone_V23.9.csv',
    sep=';', encoding='latin-1', low_memory=False
)

mask = (
    df_ademe['Tags français'].str.contains(
        'ferroviaire|TGV|TER|Intercit', case=False, na=False
    ) &
    df_ademe['Unité français'].str.contains(
        'passager|pass|pkm|voy', case=False, na=False
    )
)

df_rail = df_ademe[mask][[
    'Nom base français', 'Unité français',
    'Total poste non décomposé', 'Tags français'
]].copy()

df_rail['co2_pkm'] = df_rail['Total poste non décomposé'].str.replace(',', '.').astype(float)

summary = df_rail.groupby(
    df_rail['Nom base français'].str.extract(r'(TGV|TER|Intercités)')[0]
)['co2_pkm'].mean().reset_index()
summary.columns = ['train_type', 'co2_pkm_mean']
print(summary)

   train_type  co2_pkm_mean
0  Intercités      0.008797
1         TER      0.034818
2         TGV      0.002970


## Enrichissement du dataset

On applique les facteurs ADEME aux routes SNCF selon le type de service.
Les autres opérateurs (DB, ÖBB, SBB...) conservent le facteur Back-on-Track 
(0.014) faute de données équivalentes pour leurs réseaux nationaux.

| Type       | Facteur (kgCO2/pkm) | Mapping         |
|------------|---------------------|-----------------|
| TGV        | 0.00297             | SNCF + jour     |
| Intercités | 0.008797            | SNCF + nuit     |
| Autres     | 0.014               | Fallback        |

In [9]:
co2_map = {
    ('SNCF', 'day'):   0.00297,    # TGV
    ('SNCF', 'night'): 0.008797,   # Intercités
}
co2_default = 0.014  # Back-on-Track fallback

df['train_gco2_pkm_ademe'] = df.apply(
    lambda row: co2_map.get((row['operator'], row['service_type']), co2_default),
    axis=1
)

# Validation
print(df[df['operator'] == 'SNCF'].groupby('service_type')['train_gco2_pkm_ademe'].mean())
print(f"\nValeurs uniques : {sorted(df['train_gco2_pkm_ademe'].unique())}")

service_type
day      0.002970
night    0.008797
Name: train_gco2_pkm_ademe, dtype: float64

Valeurs uniques : [np.float64(0.00297), np.float64(0.008797), np.float64(0.014)]


## Conclusion

Le dataset enrichi contient désormais des facteurs CO2 variables pour les 
trains français (SNCF), remplaçant la constante uniforme de Back-on-Track 2022.

**Prochaine étape** : Utiliser `train_gco2_pkm_ademe` pour recalculer 
`co2_savings_kg` et en faire une vraie cible ML non triviale.

In [10]:
# Recalcul co2_savings_kg avec facteurs variables
df['train_co2_kg_ademe'] = df['distance_km'] * df['train_gco2_pkm_ademe']
df['co2_savings_kg_ademe'] = df['plane_co2_kg'] - df['train_co2_kg_ademe']

# Vérification
print(df[['operator', 'service_type', 'train_gco2_pkm_ademe', 
          'co2_savings_kg', 'co2_savings_kg_ademe']].head(10))
print(f"\nVariance co2_savings_kg original : {df['co2_savings_kg'].std():.3f}")
print(f"Variance co2_savings_kg ADEME    : {df['co2_savings_kg_ademe'].std():.3f}")

  operator service_type  train_gco2_pkm_ademe  co2_savings_kg  \
0       DB          day                 0.014           31.02   
1       DB          day                 0.014           33.32   
2       DB          day                 0.014           38.56   
3       DB          day                 0.014           37.14   
4       DB          day                 0.014           41.22   
5       DB          day                 0.014           70.28   
6       DB        night                 0.014           70.49   
7       DB          day                 0.014           69.75   
8       DB          day                 0.014           70.01   
9       DB          day                 0.014           69.73   

   co2_savings_kg_ademe  
0              31.01904  
1              33.32138  
2              38.55760  
3              37.14076  
4              41.22060  
5              70.28146  
6              70.48878  
7              69.74830  
8              70.01016  
9              69.73068 

In [11]:
sncf = df[df['operator'] == 'SNCF']
print(f"SNCF - variance original : {sncf['co2_savings_kg'].std():.3f}")
print(f"SNCF - variance ADEME    : {sncf['co2_savings_kg_ademe'].std():.3f}")
print(f"\nNombre routes SNCF : {len(sncf)}")
print(f"Nombre routes total : {len(df)}")

SNCF - variance original : 21.529
SNCF - variance ADEME    : 23.356

Nombre routes SNCF : 1333
Nombre routes total : 3678


## Conclusion

L'enrichissement ADEME améliore la variance de `co2_savings_kg` pour les 
routes SNCF (+8.5%), mais reste limité à 36% du dataset.

**Limites** :
- DB, ÖBB, SBB et les autres opérateurs conservent le facteur Back-on-Track
- Aucune source équivalente à l'ADEME n'a été trouvée pour les réseaux 
  ferroviaires allemand, autrichien et suisse

**Décision** : On conserve `duration_minutes` comme cible ML principale — 
variable non triviale, indépendante, exploitable sur l'ensemble du dataset.
`co2_savings_kg_ademe` peut servir de cible secondaire pour les routes SNCF.

## Enrichissement — Modal Split ferroviaire (Eurostat tran_hv_psmod)

Problème identifié : `co2_savings_kg` est une formule de `distance_km` → 
corrélation parfaite → pas de vraie cible ML.

Solution : ajouter la part modale ferroviaire par pays (% des voyageurs 
qui choisissent le train). Cette variable dépend des politiques de transport, 
de l'infrastructure et de la culture de chaque pays — elle est indépendante 
de la distance et apporte de la variance réelle au modèle.

Source : Eurostat tran_hv_psmod, 2023.
NB : j'ignore la période du COVID puisque conséquence exceptionnelle. Cibler 2023 puisque représentatif du comportement post-covid.

In [ ]:
df_modal = pd.read_csv(
    'data/raw/tran_hv_psmod.tsv',
    sep='\t'
)
print(df_modal.shape)
print(df_modal.columns.tolist())
df_modal.head(3)

(148, 11)
['freq,unit,vehicle,geo\\TIME_PERIOD', '2014 ', '2015 ', '2016 ', '2017 ', '2018 ', '2019 ', '2020 ', '2021 ', '2022 ', '2023 ']


,"freq,unit,vehicle,geo\TIME_PERIOD",2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,"A,PC,BUS_TOT,AT",9.9,9.9,9.9,10.0,9.1 e,8.9 e,9.2 e,8.9 e,9.5 e,9.5 e
1,"A,PC,BUS_TOT,BE",11.5 be,10.9 e,10.5 e,10.2 e,10.2 e,10.3 e,8.1 e,8.1 e,9.6 e,10.3 e
2,"A,PC,BUS_TOT,BG",15.1 e,14.6 e,14.1 e,13.1 e,12.0 e,13.0 e,8.6 e,8.0 e,10.6 e,8.8 e


In [19]:
# Filtrer uniquement la part modale ferroviaire
df_train_modal = df_modal[df_modal['vehicle'] == 'TRAIN'].copy()

# Extraire pays et valeur 2023
df_train_modal = df_train_modal[['geo\\TIME_PERIOD', '2023']].copy()
df_train_modal.columns = ['country_code', 'rail_modal_share_pct']

# Nettoyer les valeurs (enlever les flags " e", " m", etc.)
df_train_modal['rail_modal_share_pct'] = (
    df_train_modal['rail_modal_share_pct']
    .str.replace(r'[^0-9.]', '', regex=True)
    .replace('', float('nan'))
    .astype(float)
)

print(df_train_modal.dropna().sort_values('rail_modal_share_pct', ascending=False))

KeyError: 'vehicle'

In [15]:
df.to_csv(
    '/Users/louisgardet/dev/python/OBRail-MSPR2/data/processed/routes_enriched.csv',
    index=False
)
print("Dataset sauvegardé.")

Dataset sauvegardé.
